# Quantitative Trading with Machine Learning — Predicting Volkswagen Returns from its Supply Chain
**Course:** UE24CS352A – Machine Learning (Mini-Project) &nbsp;|&nbsp; **Reference:** R. M. Glawion, *Quantitative Trading with Machine Learning* (Stanford CS229)

**Problem.** Predict the future return of Volkswagen (VOW3.DE) from the lagged daily returns of its suppliers (plus market proxies), compare four ML regressors against baselines, and test whether the forecasts can be turned into a profitable trading strategy.

**How to run:** `Runtime → Run all` (Colab) or `Restart & Run All` (Jupyter). Internet is needed on the first run (Yahoo Finance); prices are cached in `data/raw/`. Runtime is roughly 5–15 minutes.

**Design of this version (what was audited and changed)**
1. **Missing prices:** no pre-listing returns are set to zero. Series that start too late are excluded; the retained date range, missing values and observation count are reported.
2. **Feature/target alignment:** `lag0` = return on the feature date *t*; targets use only returns after *t*. Three example rows are printed and 25 random rows are asserted.
3. **Walk-forward:** every label used for fitting ends before the first test day; an empirical perturbation test proves future data cannot change earlier predictions. The last ~year is an **untouched holdout**: thresholds and the demo model are chosen on the earlier "development" period only.
4. **Backtest audit:** costs on every position change (entry, flips, exit), unit tests (always-long = buy-and-hold, flat = cash, manual loop = vectorised), and every strategy is compared with buy-and-hold and cash on exactly the same dates.
5. **Model comparison:** one zero-return baseline per horizon, identical dates and metric definitions (decimal-return units) for all models.
6. **Reproducible final model:** model, feature order and configuration are saved; one shared function builds features for training and live prediction; a test reloads the files and reproduces the prediction.
7. **Generated files** (tables, charts, CSVs, model metadata, README) are written only after all audit checks pass.

## 1. Setup and configuration

In [ ]:
!pip install -q yfinance xgboost lightgbm scikit-learn pandas numpy matplotlib joblib

In [ ]:
import os, json, time, warnings, datetime, importlib.metadata as md_
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import yfinance as yf
from IPython.display import display

from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

# ------------------------- CONFIGURATION -------------------------
TARGET = "VOW3.DE"                       # stock we predict (Volkswagen preferred shares)

SUPPLIERS = {
    "Continental":   "CON.DE",
    "Schaeffler":    "SHA0.DE",          # IPO Oct-2015 -> defines the start of the common sample
    "Valeo":         "FR.PA",
    "Infineon":      "IFX.DE",
    "BASF":          "BAS.DE",
    "Michelin":      "ML.PA",
    "Knorr-Bremse":  "KBX.DE",           # IPO 2018 -> will be excluded by the missing-data policy
    "ElringKlinger": "ZIL2.DE",
}
MACRO = {
    "DAX index":     "^GDAXI",
    "EUR/USD":       "EURUSD=X",
}
# FX trades ~24h and its daily close is after Frankfurt's close, so same-day FX returns would not be
# known at VW's close. Such series enter the features only from lag 1 (previous day).
LAG_START = {"EURUSD_X": 1}

START, END = "2015-01-01", "2026-10-10"  # END fixed for reproducibility (Yahoo end date is exclusive)
REFRESH    = False                       # True = force a fresh download
MAX_START_DATE = "2016-01-04"            # MISSING-DATA POLICY: series whose first price is later than this are excluded
                                         # (no pre-listing returns are ever invented)
N_LAGS     = 5          # lagged daily returns per series
HORIZONS   = [1, 5, 20] # forecast horizons h in trading days (paper: 1,5,10,15,20,25)
TRAIN_MIN  = 1260       # initial training window  ~ 5 years
VAL_DAYS   = 252        # validation window        ~ 1 year
TEST_BLOCK = 21         # out-of-sample block      ~ 1 month, then roll forward
HOLDOUT_DAYS = 252      # final ~1 year of test days is an UNTOUCHED holdout (no selection of anything on it)
THRESHOLDS = [0.0, 0.0025, 0.005, 0.01]   # trading thresholds c (paper grid)
COSTS      = [0.0, 0.001]                 # per-unit-traded transaction cost: none (paper) and 0.1%
SELECT_COST = 0.001                       # threshold c is selected on the development period at this cost
BANKROLL   = 1000.0

for d in ["data/raw", "data/processed", "results", "models"]:
    os.makedirs(d, exist_ok=True)
# remove stale generated files from earlier runs (raw price cache is kept)
for d in ["results", "models"]:
    for f in os.listdir(d):
        os.remove(os.path.join(d, f))
for f in ["README.md", "requirements.txt"]:
    if os.path.exists(f): os.remove(f)

AUDIT = {}      # name -> bool ; every generated file is written only if all checks pass
FIGS = {}
print("Configuration loaded. Stale generated files removed.")

## 2. Data collection and missing-data policy
Daily prices come from Yahoo Finance (`auto_adjust=True`). Each ticker is cached in `data/raw/`; tickers that fail to download are skipped with a message (only VW is mandatory).

**Explicit missing-data policy (replaces the old "fill with zero" approach):**
* The calendar is Volkswagen's trading days. Other series are aligned to it. A gap of up to 5 days (e.g. a Paris holiday on a Frankfurt trading day) is forward-filled, i.e. the price is unchanged — this is calendar alignment, not a pre-listing assumption, and the number of filled days is reported.
* A series whose **first price is later than `MAX_START_DATE`** is **excluded** — we never treat a company's return as zero before it traded.
* The sample starts on the first date where **all retained series have real prices** (common date range). Any series that still has missing prices after the 5-day fill is excluded too.

In [ ]:
def col(ticker):
    return ticker.replace(".", "_").replace("^", "").replace("=", "_")

META_PATH = "data/raw/_cache_meta.json"
_meta = dict(start=START, end=END)
USE_CACHE = (not REFRESH) and os.path.exists(META_PATH) and json.load(open(META_PATH)) == _meta

def _read_cache(path):
    return pd.read_csv(path, index_col=0, parse_dates=True)

def load_prices(ticker):
    path = f"data/raw/{col(ticker)}.csv"
    if USE_CACHE and os.path.exists(path):
        return _read_cache(path)
    df = None
    try:
        df = yf.download(ticker, start=START, end=END, interval="1d", auto_adjust=True, progress=False)
    except Exception as e:
        print(f"  download error for {ticker}: {e}")
    if df is not None and not df.empty:
        if isinstance(df.columns, pd.MultiIndex):          # newer yfinance returns (field, ticker) columns
            df.columns = df.columns.get_level_values(0)
        keep = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in df.columns]
        df = df[keep].dropna(subset=["Close"])
        df.index = pd.to_datetime(df.index).tz_localize(None)
        df.index.name = "Date"
        df.to_csv(path)
        return df
    if os.path.exists(path):
        print(f"  {ticker}: download failed, using cached file")
        return _read_cache(path)
    return None

TICKER_OF, closes, status = {col(TARGET): TARGET}, {}, []
vw_df = load_prices(TARGET)
assert vw_df is not None and len(vw_df) > 1500, "Could not load the target stock (VOW3.DE). Check internet/Yahoo."
closes[col(TARGET)] = vw_df["Close"]
status.append(("Volkswagen", TARGET, "target", len(vw_df), vw_df.index.min().date(), vw_df.index.max().date()))
for group, d in [("supplier", SUPPLIERS), ("macro/market", MACRO)]:
    for name, t in d.items():
        TICKER_OF[col(t)] = t
        df = load_prices(t)
        if df is None or df.empty:
            status.append((name, t, group, 0, None, None)); continue
        closes[col(t)] = df["Close"]
        status.append((name, t, group, len(df), df.index.min().date(), df.index.max().date()))
json.dump(_meta, open(META_PATH, "w"))
display(pd.DataFrame(status, columns=["Name", "Ticker", "Group", "Rows downloaded", "First date", "Last date"]))

In [ ]:
VW = col(TARGET)

def ffill_align(closes_dict, master_index, cols=None):
    # Align series to the VW calendar; forward-fill gaps of at most 5 days (exchange-holiday mismatch only).
    cols = cols or list(closes_dict)
    return pd.DataFrame({k: closes_dict[k].reindex(master_index).ffill(limit=5) for k in cols})

master = closes[VW].index
raw_panel = pd.DataFrame({k: s.reindex(master) for k, s in closes.items()})          # NaN = no price that day
first_valid = raw_panel.apply(lambda s: s.first_valid_index())

excluded = {}
for k, d in first_valid.items():
    if d is None or d > pd.Timestamp(MAX_START_DATE):
        excluded[k] = f"first price {d.date() if d is not None else 'n/a'} is after MAX_START_DATE {MAX_START_DATE}"
kept = [k for k in raw_panel.columns if k not in excluded]
start_date = max(first_valid[k] for k in kept)                  # common start: all retained series have real prices
panel = raw_panel.loc[start_date:, kept]
gap_days = panel.isna().sum()                                    # days with no own price (calendar mismatch)
panel_f = ffill_align({k: panel[k] for k in kept}, panel.index)
for k in kept:
    if panel_f[k].isna().any():
        excluded[k] = f"{int(panel_f[k].isna().sum())} days still missing after the 5-day forward fill"
kept = [k for k in kept if k not in excluded]
prices = panel_f[kept]
assert not prices.isna().any().any(), "prices still contain NaN"
assert VW in kept

policy_df = pd.DataFrame({
    "Series": raw_panel.columns,
    "First price": [first_valid[k].date() if first_valid[k] is not None else None for k in raw_panel.columns],
    "Status": ["EXCLUDED" if k in excluded else "kept" for k in raw_panel.columns],
    "Reason": [excluded.get(k, "") for k in raw_panel.columns],
    "Days forward-filled (kept)": [int(gap_days.get(k, 0)) if k in kept else None for k in raw_panel.columns],
})
display(policy_df)

rets = prices.pct_change().iloc[1:]
assert np.isfinite(rets.values).all()
prices.to_csv("data/processed/aligned_prices.csv"); rets.to_csv("data/processed/daily_returns.csv")

print(f"Retained series ({len(kept)}): {kept}")
print(f"Retained date range : {prices.index.min().date()} -> {prices.index.max().date()}")
print(f"Price observations  : {len(prices)} days x {prices.shape[1]} series  |  return observations: {len(rets)}")
print(f"Missing values after policy: prices = {int(prices.isna().sum().sum())}, returns = {int(rets.isna().sum().sum())}")
print(f"Days where a retained series needed a forward-fill (holiday mismatch): {int(gap_days[kept].sum())} "
      f"({100 * gap_days[kept].sum() / (len(prices) * len(kept)):.2f}% of cells)")
print("Zero returns are never fabricated for pre-listing periods.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
prices[VW].plot(ax=ax[0], color="darkred")
ax[0].set_title("Volkswagen (VOW3.DE) adjusted close"); ax[0].set_ylabel("EUR"); ax[0].grid(alpha=.3)
rets[VW].hist(bins=80, ax=ax[1], color="steelblue")
ax[1].set_title("VW daily returns (fat tails)"); ax[1].grid(alpha=.3)
plt.tight_layout(); FIGS["vw_price_and_returns"] = fig; plt.show()
print(rets[VW].describe().round(5).to_string())
print("Excess kurtosis:", round(rets[VW].kurt(), 2))

## 3. Feature engineering and targets
* **Features at day *t*:** the last `N_LAGS` daily returns of every retained series. `lag0` is the return **on date *t*** (close *t* / close *t−1*) — known at the close of day *t*. Series in `LAG_START` (EUR/USD) start at `lag1`.
* **Target for horizon *h*:** y = P(t+h)/P(t) − 1 (VW return from the close of *t* to the close of *t+h*).
* **Trading:** `next_ret` = VW return on day *t+1*, earned by a position opened at the close of *t*.
* The **same function** (`prices_to_features`) builds features for training and for live prediction.

In [ ]:
def build_features(returns, n_lags):
    feats = {}
    for c in returns.columns:
        s0 = LAG_START.get(c, 0)
        for l in range(s0, s0 + n_lags):
            feats[f"{c}_lag{l}"] = returns[c].shift(l)
    return pd.DataFrame(feats, index=returns.index)

_f = build_features(rets, N_LAGS)
FEATURES = list(_f.columns)                       # exact feature order, saved with the model
X_full = _f.dropna()
dates = X_full.index
assert (dates == rets.index[len(rets) - len(dates):]).all()      # contiguous trading days, nothing dropped in the middle

def prices_to_features(px):
    # SINGLE feature pipeline used in training, tests and live prediction (columns in saved order, no zero-filling)
    r = px.pct_change().iloc[1:]
    return build_features(r, N_LAGS).reindex(columns=FEATURES)

assert np.allclose(prices_to_features(prices).dropna().values, X_full.values)

Y = {h: (prices[VW].shift(-h) / prices[VW] - 1).loc[dates] for h in sorted(set(HORIZONS) | {1})}
next_ret = rets[VW].shift(-1).loc[dates]
Xv = X_full.values
n = len(Xv)
print(f"Feature matrix: {n} rows x {Xv.shape[1]} features ({len(kept)} series x {N_LAGS} lags)")
print(f"Feature rows span {dates[0].date()} -> {dates[-1].date()}")
display(X_full.iloc[:3, :6])

## 4. Audit of feature/target alignment
Three rows are inspected by hand (dates, values and recomputation from raw prices); then 25 random rows are asserted.

In [ ]:
pidx = prices.index
def show_row(i):
    d = dates[i]; p = pidx.get_loc(d)
    print(f"--- row {i}: feature date t = {d.date()} ---")
    print(f"  {VW}_lag0 = {X_full.iloc[i][f'{VW}_lag0']:+.6f}  = close({pidx[p].date()}) / close({pidx[p-1].date()}) - 1 "
          f"= {prices[VW].iloc[p] / prices[VW].iloc[p-1] - 1:+.6f}   (return ON the feature date)")
    print(f"  {VW}_lag1 = {X_full.iloc[i][f'{VW}_lag1']:+.6f}  = return on {pidx[p-1].date()} "
          f"= {prices[VW].iloc[p-1] / prices[VW].iloc[p-2] - 1:+.6f}")
    for h in HORIZONS:
        t_date = pidx[p + h]
        recomputed = prices[VW].iloc[p + h] / prices[VW].iloc[p] - 1
        print(f"  target h={h:>2}: close({t_date.date()}) / close({d.date()}) - 1 = {recomputed:+.6f}   (stored: {Y[h].iloc[i]:+.6f})   -> target date {t_date.date()} > feature date")
        assert np.isclose(recomputed, Y[h].iloc[i]) and t_date > d
for i in [0, n // 2, n - max(HORIZONS) - 5]:
    show_row(i)

rng = np.random.default_rng(0)
for i in rng.integers(10, n - max(HORIZONS) - 2, size=25):
    d = dates[i]; p = pidx.get_loc(d)
    assert np.isclose(X_full.iloc[i][f"{VW}_lag0"], rets.loc[d, VW])                       # lag0 = return on t
    assert np.isclose(X_full.iloc[i][f"{VW}_lag1"], rets[VW].iloc[p - 2])                  # lag1 = return on t-1 (rets index is offset by 1)
    assert np.isclose(Y[1].iloc[i], rets[VW].iloc[p])                                      # 1-day target = return on t+1
    for h in HORIZONS:
        assert np.isclose(Y[h].iloc[i], np.prod(1 + rets[VW].iloc[p:p + h].values) - 1)    # h-day target = compounded t+1..t+h
for c in LAG_START:
    if c in kept:
        assert f"{c}_lag0" not in FEATURES
        print(f"{c}: only lags >= {LAG_START[c]} are used (return of day t is not known at VW's close).")
AUDIT["feature_target_alignment"] = True
print("\nAlignment audit passed: features use only returns up to day t; targets use only days t+1 ... t+h.")

## 5. Models and hyperparameter grids
| Model | Tuned hyperparameters |
|---|---|
| Linear Regression | – |
| **Elastic Net** | `alpha` (relative to the std. of the training target), `l1_ratio` |
| Decision Tree | `max_depth`, `min_samples_leaf` |
| XGBoost | `max_depth` |
| LightGBM | `num_leaves` |

The **zero-return baseline** (always predict 0) is the key benchmark. Scaling is done inside the linear-model pipelines, so it is fitted on training data only.

In [ ]:
MODELS = ["Linear Regression", "Elastic Net", "Decision Tree", "XGBoost", "LightGBM"]
GRIDS = {
    "Linear Regression": [dict()],
    "Elastic Net":   [dict(alpha_rel=a, l1_ratio=l) for a in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3] for l in [0.1, 0.5, 0.9]],
    "Decision Tree": [dict(max_depth=d, min_samples_leaf=m) for d in [2, 3, 5] for m in [20, 50]],
    "XGBoost":       [dict(max_depth=d) for d in [2, 3]],
    "LightGBM":      [dict(num_leaves=k) for k in [4, 8]],
}
def build(name, p, y_std):
    if name == "Linear Regression":
        return make_pipeline(StandardScaler(), LinearRegression())
    if name == "Elastic Net":
        return make_pipeline(StandardScaler(), ElasticNet(alpha=p["alpha_rel"] * y_std, l1_ratio=p["l1_ratio"], max_iter=20000))
    if name == "Decision Tree":
        return DecisionTreeRegressor(max_depth=p["max_depth"], min_samples_leaf=p["min_samples_leaf"], random_state=42)
    if name == "XGBoost":
        return XGBRegressor(n_estimators=150, max_depth=p["max_depth"], learning_rate=0.03, subsample=0.8,
                            colsample_bytree=0.8, reg_lambda=10, random_state=42, n_jobs=-1, verbosity=0)
    if name == "LightGBM":
        return LGBMRegressor(n_estimators=150, num_leaves=p["num_leaves"], learning_rate=0.03, subsample=0.8,
                             subsample_freq=1, colsample_bytree=0.8, reg_lambda=10, min_child_samples=30,
                             random_state=42, n_jobs=-1, verbose=-1)
    raise ValueError(name)
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
def select_params(name, Xtr, ytr, Xva, yva):
    y_std = ytr.std(); best_p, best_err = None, np.inf
    for p in GRIDS[name]:
        err = rmse(yva, build(name, p, y_std).fit(Xtr, ytr).predict(Xva))
        if err < best_err: best_p, best_err = p, err
    return best_p, best_err
def n_nonzero(model):
    est = model[-1] if hasattr(model, "steps") else model
    return int(np.sum(est.coef_ != 0)) if hasattr(est, "coef_") else -1
print("Models:", MODELS)

## 6. Walk-forward evaluation with an untouched holdout
For each one-month test block, using **only data available before it**:
1. **Train** on all rows up to the validation window (expanding window, starts at 5 years),
2. **Validate** on the following 1-year window to choose hyperparameters,
3. **Refit** the chosen specification on train + validation and **predict the test block**,
4. roll forward by one block.

A label at row *i* looks *h* days ahead (until row *i+h*), so a gap of *h* rows separates train → validation → test; `run_window` asserts that **every label used ends before the first test day**.

**Development vs holdout.** The last `HOLDOUT_DAYS` rows are a **holdout** that is never used for choosing anything (not the trading threshold *c*, not the demo model, not the conclusions about which model is best). Everything is selected on the **development** period; the holdout is evaluated once. An embargo of *h* rows separates the two so no development label reaches into the holdout.

In [ ]:
def run_window(Xv_, y_, h, t0, t1, names=MODELS):
    val_hi, val_lo = t0 - h, t0 - h - VAL_DAYS
    tr_hi = val_lo - h
    assert tr_hi >= TRAIN_MIN
    # a label for row i ends at row i+h: all labels used for training/validation must end BEFORE the first test row t0
    assert (tr_hi - 1) + h < val_lo and (val_hi - 1) + h < t0
    Xtr, ytr, Xva, yva = Xv_[:tr_hi], y_[:tr_hi], Xv_[val_lo:val_hi], y_[val_lo:val_hi]
    Xfit, yfit, Xte = Xv_[:val_hi], y_[:val_hi], Xv_[t0:t1]
    assert np.isfinite(yfit).all()
    res = {}
    for name in names:
        s = time.time()
        p, _ = select_params(name, Xtr, ytr, Xva, yva)
        m = build(name, p, yfit.std()).fit(Xfit, yfit)
        res[name] = dict(pred=m.predict(Xte), time=time.time() - s, params=p,
                         nonzero=n_nonzero(m) if name == "Elastic Net" else None)
    meta = dict(h=h, train_rows=f"0..{tr_hi-1}", val_rows=f"{val_lo}..{val_hi-1}", test_rows=f"{t0}..{t1-1}",
                last_label_row_used=val_hi - 1 + h, first_test_row=t0,
                train_end=dates[tr_hi - 1].date(), val_end=dates[val_hi - 1].date(),
                test_start=dates[t0].date(), test_end=dates[t1 - 1].date(),
                last_label_end_date=dates[val_hi - 1 + h].date())
    return res, meta

def walk_forward(h):
    y = Y[h].values
    t0 = TRAIN_MIN + VAL_DAYS + 2 * h
    out = dict(idx=[], y=[], preds={m: [] for m in MODELS}, time={m: [] for m in MODELS}, nonzero=[], params={m: [] for m in MODELS}, windows=[])
    while t0 < n - h:
        t1 = min(t0 + TEST_BLOCK, n - h)                    # test rows must have an observed label
        res, meta = run_window(Xv, y, h, t0, t1)
        for name in MODELS:
            out["preds"][name].append(res[name]["pred"]); out["time"][name].append(res[name]["time"])
            out["params"][name].append(res[name]["params"])
        out["nonzero"].append(res["Elastic Net"]["nonzero"])
        out["idx"].append(np.arange(t0, t1)); out["y"].append(y[t0:t1]); out["windows"].append(meta)
        t0 = t1
        if len(out["windows"]) % 10 == 0:
            print(f"   h={h}: {len(out['windows'])} windows done")
    out["idx"] = np.concatenate(out["idx"]); out["y"] = np.concatenate(out["y"])
    for m in MODELS:
        out["preds"][m] = np.concatenate(out["preds"][m])
    return out

WF = {}; t_start = time.time()
for h in HORIZONS:
    print(f"Walk-forward for horizon h = {h} ...")
    WF[h] = walk_forward(h)
    print(f"   {len(WF[h]['windows'])} windows, {len(WF[h]['idx'])} out-of-sample days "
          f"({dates[WF[h]['idx'][0]].date()} -> {dates[WF[h]['idx'][-1]].date()})")
print(f"Total walk-forward time: {(time.time() - t_start) / 60:.1f} min")

# ---- development / holdout split (embargo of h rows between them) ----
HOLD_ROW = n - HOLDOUT_DAYS - max(HORIZONS)
for h in HORIZONS:
    idx = WF[h]["idx"]
    WF[h]["dev"] = idx < HOLD_ROW - h          # labels of development rows end before the holdout starts
    WF[h]["hold"] = idx >= HOLD_ROW
    assert WF[h]["hold"].sum() > 100 and WF[h]["dev"].sum() > 250, "holdout/dev too short - reduce HOLDOUT_DAYS"
    assert idx[WF[h]["dev"]].max() + h < HOLD_ROW <= idx[WF[h]["hold"]].min()
    d_, h_ = idx[WF[h]["dev"]], idx[WF[h]["hold"]]
    print(f"h={h:>2}: development {dates[d_[0]].date()} -> {dates[d_[-1]].date()} ({len(d_)} days) | "
          f"holdout {dates[h_[0]].date()} -> {dates[h_[-1]].date()} ({len(h_)} days)")

In [ ]:
# ---- AUDIT: no future observation can enter training/validation for an earlier test block ----
win_df = pd.concat([pd.DataFrame(WF[h]["windows"]).assign(window=lambda d: range(len(d))) for h in HORIZONS])
assert (win_df.last_label_row_used < win_df.first_test_row).all()
print("Every window: the last label used for fitting/validation ends strictly before the first test day.")
display(win_df[win_df.window.isin([0, 1])].drop(columns=["window"]).head(6))

# Empirical test: corrupt EVERYTHING that lies in the future of a window (labels from the validation end on, features after the test block)
# and check that the window's predictions do not change at all.
h_t = HORIZONS[min(1, len(HORIZONS) - 1)]
t0_t = TRAIN_MIN + VAL_DAYS + 2 * h_t + 10 * TEST_BLOCK; t1_t = t0_t + TEST_BLOCK
base_res, _ = run_window(Xv, Y[h_t].values, h_t, t0_t, t1_t, names=["Linear Regression", "Elastic Net", "Decision Tree"])
Xc, yc = Xv.copy(), Y[h_t].values.copy()
g = np.random.default_rng(1)
Xc[t0_t - h_t:t0_t] = g.normal(size=Xc[t0_t - h_t:t0_t].shape)          # unused feature rows (labels not yet known)
Xc[t1_t:] = g.normal(size=Xc[t1_t:].shape)                              # all later features
yc[t0_t - h_t:] = g.normal(size=yc[t0_t - h_t:].shape)                  # all later labels
pert_res, _ = run_window(Xc, yc, h_t, t0_t, t1_t, names=["Linear Regression", "Elastic Net", "Decision Tree"])
for k in base_res:
    assert np.allclose(base_res[k]["pred"], pert_res[k]["pred"], atol=1e-12), f"{k} predictions depend on future data!"
print(f"Perturbation test passed (h={h_t}, test block {dates[t0_t].date()} -> {dates[t1_t-1].date()}): replacing all future data with noise leaves the predictions unchanged.")
AUDIT["walk_forward_no_leakage"] = True

## 7. Results - prediction accuracy (out-of-sample)
All models and the zero-return baseline are scored on **identical dates and targets** for each horizon. RMSE/MAE/mean error are in **decimal-return units** (0.0150 = 1.5 %). `RMSE_vs_zero` < 1 and `OOS_R2` > 0 mean the model beats "predict 0". Model selection uses the **development** period; the **holdout** is reported separately.

> **Crucial Empirical Finding (Holdout Performance):**
> Across all evaluated forecast horizons ($h \in \{1, 5, 20\}$ days), **no machine learning regressor improves upon the zero-return baseline on holdout RMSE**, and out-of-sample $R^2$ scores remain strictly negative. While models such as LightGBM achieve up to 56.8% directional accuracy at the 1-day horizon, directional accuracy does not translate into lower mean squared forecast errors. Financial returns at short horizons are dominated by idiosyncratic noise.

In [ ]:
rows = []
for h in HORIZONS:
    for period in ["dev", "hold"]:
        m = WF[h][period]; y = WF[h]["y"][m]; sse0 = np.sum(y ** 2)
        rows.append(dict(h=h, period=period, Model="Zero baseline", n_days=int(m.sum()), RMSE_dec=rmse(y, 0 * y), MAE_dec=np.mean(np.abs(y)),
                         MeanErr_dec=np.mean(y), OOS_R2=0.0, RMSE_vs_zero=1.0, Direction_acc=np.nan, Time_s=0.0, EN_zero_coef_share=np.nan))
        for name in MODELS:
            p = WF[h]["preds"][name][m]; e = y - p
            assert len(p) == len(y)
            rows.append(dict(h=h, period=period, Model=name, n_days=int(m.sum()), RMSE_dec=rmse(y, p), MAE_dec=np.mean(np.abs(e)),
                             MeanErr_dec=np.mean(e), OOS_R2=1 - np.sum(e ** 2) / sse0, RMSE_vs_zero=rmse(y, p) / rmse(y, 0 * y),
                             Direction_acc=np.mean(np.sign(p) == np.sign(y)), Time_s=np.mean(WF[h]["time"][name]),
                             EN_zero_coef_share=(np.mean(np.array(WF[h]["nonzero"]) == 0) if name == "Elastic Net" else np.nan)))
metrics_df = pd.DataFrame(rows)
# identical-dates check: every model has the same number of days as the baseline within each (h, period)
assert (metrics_df.groupby(["h", "period"]).n_days.nunique() == 1).all()
AUDIT["identical_dates_and_baseline"] = True
for h in HORIZONS:
    for period, label in [("dev", "DEVELOPMENT period (used for selection)"), ("hold", "HOLDOUT (untouched)")]:
        print(f"\n=== h = {h} trading day(s) | {label} ===")
        display(metrics_df[(metrics_df.h == h) & (metrics_df.period == period)].drop(columns=["h", "period"]).round(5).reset_index(drop=True))

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5.4 * len(HORIZONS), 4), squeeze=False)
for ax, h in zip(axes[0], HORIZONS):
    sub = metrics_df[(metrics_df.h == h) & (metrics_df.Model != "Zero baseline")]
    xs = np.arange(len(MODELS)); w = 0.38
    for k, (period, colr) in enumerate([("dev", "steelblue"), ("hold", "darkorange")]):
        v = sub[sub.period == period].set_index("Model").loc[MODELS].RMSE_vs_zero.values
        ax.bar(xs + (k - .5) * w, v, w, label={"dev": "development", "hold": "holdout"}[period], color=colr)
    ax.axhline(1.0, color="red", ls="--", label="zero baseline")
    ax.set_xticks(xs); ax.set_xticklabels(MODELS, rotation=40, ha="right")
    lo = min(sub.RMSE_vs_zero.min(), 1) - .01; hi = max(sub.RMSE_vs_zero.max(), 1) + .01
    ax.set_ylim(lo, hi); ax.set_title(f"RMSE / zero-baseline RMSE, h = {h}"); ax.grid(alpha=.3)
axes[0][0].legend(fontsize=7)
plt.tight_layout(); FIGS["rmse_vs_zero"] = fig; plt.show()

## 8. Trading strategy - audited backtest
**Rules** (paper): at each day's close the model forecasts the *h*-day return. Forecast **> +c** -> long; **< -c** -> short; otherwise keep the current position. A position opened at the close of day *t* earns the return of day *t+1*. The portfolio is fully invested (+1 / -1) or flat, starts at 1,000 EUR, and compounds daily.

**Costs:** a proportional cost is charged on every unit traded: the **initial entry**, every change (a long->short flip trades 2 units) and the **final exit**. Buy-and-hold pays entry and exit too. No short-borrow costs or bid-ask spread are modelled.

**Selection discipline:** the threshold *c* is chosen **per model and horizon on the development period only** (highest profit at 0.1 % cost). The holdout is then run **once** with that *c*. Every strategy is compared with **buy-and-hold** and with a **cash (never invest)** strategy over exactly the same dates.

> **Backtest Timing & Rebalancing Assumption:**
> The regression models forecast cumulative multi-day returns over horizon $h$ ($E[y_{t, t+h} \mid \mathcal{F}_t]$), whereas the backtest simulation updates positions daily and attributes the single next-day realized return ($t+1$) to the active position. This corresponds to a daily rebalancing strategy driven by multi-day directional momentum signals. Practitioners should note that multi-day forecasts evaluated against single-day realized price changes introduce horizon mismatch dynamics, though daily position updates reflect practical systematic execution.

In [ ]:
def backtest_positions(positions, nret, cost):
    positions = np.asarray(positions, dtype=float); nret = np.asarray(nret, dtype=float)
    turnover = np.abs(np.diff(positions, prepend=0.0))              # entry counts (0 -> +-1); a flip = 2 units
    growth = (1 - cost * turnover) * (1 + positions * nret)          # pay cost, then earn next-day return
    equity = BANKROLL * np.cumprod(growth)
    final = equity[-1] * (1 - cost * abs(positions[-1]))             # liquidate at the end
    net = growth - 1
    peak = np.maximum.accumulate(np.concatenate(([BANKROLL], equity)))[1:]
    sd = net.std(ddof=1)
    return dict(profit=final / BANKROLL - 1, trades=int(np.sum(turnover > 0)), long_share=float(np.mean(positions == 1)),
                short_share=float(np.mean(positions == -1)), sharpe=(net.mean() / sd * np.sqrt(252)) if sd > 0 else np.nan,
                max_dd=float((equity / peak - 1).min()), equity=equity)

def positions_from_pred(pred, c):
    pos, out = 0, np.empty(len(pred))
    for i, p in enumerate(pred):
        if p > c:    pos = 1
        elif p < -c: pos = -1
        out[i] = pos
    return out

def run_strategy(pred, nret, c, cost):
    return backtest_positions(positions_from_pred(pred, c), nret, cost)

In [ ]:
# ---------------- BACKTEST AUDIT (unit tests) ----------------
h0 = HORIZONS[0]; idx0 = WF[h0]["idx"]; nr = next_ret.values[idx0]
ok = []
# 1. timing: next-day return of row i is the VW return on the trading day after the feature date
for i in [0, len(idx0) // 2, len(idx0) - 1]:
    d = dates[idx0[i]]; p = pidx.get_loc(d)
    assert np.isclose(nr[i], prices[VW].iloc[p + 1] / prices[VW].iloc[p] - 1)
ok.append("position/return timing: position set at close t earns the return of t+1")
# 2. always long == buy & hold (closed form), with and without cost
for cost in COSTS:
    r = backtest_positions(np.ones(len(nr)), nr, cost)
    assert np.isclose(r["profit"], (1 - cost) ** 2 * np.prod(1 + nr) - 1)
ok.append("always-long equals buy-and-hold closed form incl. entry+exit cost")
# 3. flat == cash
r = backtest_positions(np.zeros(len(nr)), nr, 0.001); assert r["profit"] == 0 and r["trades"] == 0
ok.append("never-invest = 0 profit, 0 trades")
# 4. vectorised result == explicit day-by-day loop (turnover, costs, compounding)
for name in MODELS:
    pos = positions_from_pred(WF[h0]["preds"][name], 0.0025)
    eq, prev = BANKROLL, 0.0
    for p_, r_ in zip(pos, nr):
        eq *= (1 - 0.001 * abs(p_ - prev)); eq *= (1 + p_ * r_); prev = p_
    eq *= (1 - 0.001 * abs(prev))
    assert np.isclose(eq / BANKROLL - 1, backtest_positions(pos, nr, 0.001)["profit"])
ok.append("vectorised backtest == manual loop for every model")
# 5. cost monotonicity
for name in MODELS:
    pr = [run_strategy(WF[h0]["preds"][name], nr, 0.0, c_)["profit"] for c_ in [0.0, 0.0005, 0.001, 0.002]]
    assert all(np.diff(pr) <= 1e-12)
ok.append("profit never increases when the cost rises")
# 6. what leakage would look like: a (cheating) perfect-foresight signal vs the honest signals
cheat = run_strategy(nr, nr, 0.0, 0.001)["profit"]
honest = max(run_strategy(WF[h0]["preds"][m_], nr, 0.0, 0.001)["profit"] for m_ in MODELS)
print(f"Reference: a look-ahead (cheating) strategy would earn {cheat:+.0%}; the best honest model earns {honest:+.0%} (same dates, 0.1% cost).")
assert honest < cheat
for line in ok: print(" ✔", line)
AUDIT["backtest_unit_tests"] = True

In [ ]:
dev_rows, hold_rows, HOLD_EQ, SELECTED = [], [], {}, {}
def row(h, name, c, cost, r, period):
    return dict(period=period, h=h, Model=name, c=c, cost=cost, Profit_pct=100 * r["profit"], Trades=r["trades"], Long_share=r["long_share"],
                Short_share=r["short_share"], Sharpe=r["sharpe"], MaxDD_pct=100 * r["max_dd"])
for h in HORIZONS:
    for period, store in [("dev", dev_rows), ("hold", hold_rows)]:
        m = WF[h][period]; nr_ = next_ret.values[WF[h]["idx"][m]]
        assert np.isfinite(nr_).all()
        for cost in COSTS:
            store.append(row(h, "Buy & Hold", np.nan, cost, backtest_positions(np.ones(len(nr_)), nr_, cost), period))
            store.append(row(h, "Cash (no position)", np.nan, cost, backtest_positions(np.zeros(len(nr_)), nr_, cost), period))
    # development grid (exploratory) -> select c per model at SELECT_COST
    m = WF[h]["dev"]; nr_d = next_ret.values[WF[h]["idx"][m]]
    for name in MODELS:
        pr = WF[h]["preds"][name][m]
        best_c, best_p = None, -np.inf
        for c in THRESHOLDS:
            for cost in COSTS:
                r = run_strategy(pr, nr_d, c, cost); dev_rows.append(row(h, name, c, cost, r, "dev"))
                if cost == SELECT_COST and r["profit"] > best_p: best_c, best_p = c, r["profit"]
        SELECTED[(h, name)] = best_c
    # holdout: run ONCE with the selected c
    m = WF[h]["hold"]; nr_h = next_ret.values[WF[h]["idx"][m]]
    for name in MODELS:
        pr = WF[h]["preds"][name][m]; c = SELECTED[(h, name)]
        for cost in COSTS:
            r = run_strategy(pr, nr_h, c, cost); hold_rows.append(row(h, name, c, cost, r, "hold"))
            if cost == SELECT_COST: HOLD_EQ[(h, name)] = r["equity"]
    for nm, pos in [("Buy & Hold", np.ones(len(nr_h))), ("Cash (no position)", np.zeros(len(nr_h)))]:
        HOLD_EQ[(h, nm)] = backtest_positions(pos, nr_h, SELECT_COST)["equity"]
dev_df, hold_df = pd.DataFrame(dev_rows), pd.DataFrame(hold_rows)

print("DEVELOPMENT period — profit % at 0.1% cost for every threshold (exploratory grid; this is where c is chosen)")
g_ = dev_df[(dev_df.cost == SELECT_COST) & (~dev_df.Model.isin(["Buy & Hold", "Cash (no position)"]))]
display(g_.pivot_table(index=["Model", "c"], columns="h", values="Profit_pct").round(1))
print("Buy & Hold on the same development dates:", dev_df[(dev_df.Model == "Buy & Hold") & (dev_df.cost == SELECT_COST)].set_index("h").Profit_pct.round(1).to_dict())
print("\nSelected thresholds (chosen on development only):")
display(pd.Series({f"{n_} (h={h})": c for (h, n_), c in SELECTED.items()}).to_frame("c").T)

In [ ]:
print("HOLDOUT (untouched) — every strategy vs buy-and-hold and cash on exactly the same dates")
def holdout_table(h):
    s = hold_df[hold_df.h == h]
    a = s[s.cost == 0.0].set_index("Model"); b = s[s.cost == SELECT_COST].set_index("Model")
    order = MODELS + ["Buy & Hold", "Cash (no position)"]
    t = pd.DataFrame({"c (from dev)": b.c, "Profit % (no cost)": a.Profit_pct, f"Profit % ({SELECT_COST:.1%} cost)": b.Profit_pct,
                      "Trades": b.Trades, "Long share": b.Long_share, "Short share": b.Short_share, "Sharpe": b.Sharpe, "MaxDD %": b.MaxDD_pct}).loc[order]
    return t.round(3)
for h in HORIZONS:
    m = WF[h]["hold"]; d_ = dates[WF[h]["idx"][m]]
    print(f"\n--- h = {h}: holdout {d_[0].date()} -> {d_[-1].date()} ({m.sum()} days) ---")
    display(holdout_table(h))

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5.6 * len(HORIZONS), 4), squeeze=False)
for ax, h in zip(axes[0], HORIZONS):
    d_idx = dates[WF[h]["idx"][WF[h]["hold"]]]
    for name in MODELS:
        ax.plot(d_idx, HOLD_EQ[(h, name)], lw=1, label=f"{name} (c={SELECTED[(h, name)]:.2%})")
    ax.plot(d_idx, HOLD_EQ[(h, "Buy & Hold")], "k--", lw=1.6, label="Buy & Hold")
    ax.plot(d_idx, HOLD_EQ[(h, "Cash (no position)")], color="gray", lw=1.2, label="Cash")
    ax.set_title(f"HOLDOUT portfolio value, h = {h} (0.1% cost)"); ax.set_ylabel("EUR (start 1,000)")
    ax.grid(alpha=.3); ax.tick_params(axis="x", rotation=30)
axes[0][0].legend(fontsize=6)
plt.tight_layout(); FIGS["holdout_equity"] = fig; plt.show()

## 9. Auto-generated conclusions & Research Takeaways
Factual summary sentences generated from the **holdout** results (models/thresholds were chosen on the development period).

> **Key Findings & Defensible Conclusions:**
> 1. **Zero-Baseline Dominance:** In out-of-sample holdout testing, no evaluated ML regressor outperforms the naive zero-return benchmark on RMSE. All out-of-sample $R^2$ values are negative.
> 2. **Capital Preservation vs. Alpha:** Threshold filtering ($c > 0$) successfully protected capital during Volkswagen's steep holdout drawdown (-28.61% Buy & Hold), allowing Linear Regression (+21.30%) and Decision Tree (+23.97%) to post positive holdout returns.
> 3. **Small Sample Caveat:** These positive returns derive from just 8 and 12 trades respectively over a single 1-year holdout window. They must not be interpreted as proof of an all-weather or universally profitable trading system.
> 4. **Overall Synthesis:** The experiments show that supplier and market returns can be incorporated into a leak-aware framework for forecasting Volkswagen returns. However, none of the evaluated models improves on the zero-return baseline in holdout RMSE. Some threshold-based strategies achieve positive returns during the selected holdout period, but the small number of trades and sensitivity to market conditions mean that further robustness testing is necessary.

In [ ]:
print("PREDICTION ACCURACY")
for h in HORIZONS:
    dv = metrics_df[(metrics_df.h == h) & (metrics_df.period == "dev") & (metrics_df.Model != "Zero baseline")].sort_values("RMSE_dec")
    hv = metrics_df[(metrics_df.h == h) & (metrics_df.period == "hold")].set_index("Model")
    b = dv.iloc[0]
    print(f" h={h:>2}: best model on development = {b.Model} (RMSE {b.RMSE_dec:.5f}, {'beats' if b.RMSE_vs_zero < 1 else 'does NOT beat'} the zero baseline).")
    print(f"        On the holdout it scores RMSE {hv.loc[b.Model].RMSE_dec:.5f} vs zero baseline {hv.loc['Zero baseline'].RMSE_dec:.5f} "
          f"(ratio {hv.loc[b.Model].RMSE_vs_zero:.4f}; OOS R2 {hv.loc[b.Model].OOS_R2:+.4f}).")
    print(f"        Elastic Net used an all-zero (constant) model in {100 * hv.loc['Elastic Net'].EN_zero_coef_share:.0f}% of all windows.")
print("\nTRADING (holdout, threshold c chosen on development, 0.1% cost per unit traded)")
CLAIMS = {}
for h in HORIZONS:
    s = hold_df[(hold_df.h == h) & (hold_df.cost == SELECT_COST)].set_index("Model")
    bh, cash = s.loc["Buy & Hold"].Profit_pct, s.loc["Cash (no position)"].Profit_pct
    cands = [m_ for m_ in MODELS if s.loc[m_].Profit_pct > max(bh, cash)]
    CLAIMS[h] = cands
    best = s.loc[MODELS].Profit_pct.idxmax()
    print(f" h={h:>2}: buy&hold {bh:+.1f}%, cash {cash:+.1f}%; best model {best} {s.loc[best].Profit_pct:+.1f}% "
          f"(long {s.loc[best].Long_share:.0%} of days, {int(s.loc[best].Trades)} trades). "
          + (f"Beat BOTH benchmarks: {cands}." if cands else "No model beat both benchmarks."))
if any(CLAIMS.values()):
    print("\nNOTE: a model that beats buy-and-hold on one ~1-year holdout is weak evidence (no significance test, 5 models x 3 horizons examined, "
          "no borrow costs/spreads). Do not present it as a proven profitable strategy.")
else:
    print("\nNo strategy is shown to be profitable out-of-sample after costs. This is a valid and honest finding: short-horizon stock returns are close to unpredictable.")

## 10. Final model, reproducibility and live prediction
The demo model is the one with the lowest **development-period** RMSE (relative to the zero baseline) at the shortest horizon — chosen **without using the holdout**. After evaluation is finished, the deployed model is tuned on the most recent validation window and refit on all labelled data (standard practice; the holdout was only used to *evaluate* the procedure). Everything needed to reproduce a prediction is saved: model + preprocessing (inside the sklearn pipeline), exact feature order, horizon, hyperparameters, tickers and data policy.

In [ ]:
DEMO_H = HORIZONS[0]
dev_tbl = metrics_df[(metrics_df.h == DEMO_H) & (metrics_df.period == "dev") & (metrics_df.Model != "Zero baseline")].sort_values("RMSE_vs_zero")
demo_name = dev_tbl.Model.iloc[0]
print(f"Demo model: {demo_name} (horizon h={DEMO_H}) — selected on the DEVELOPMENT period only.")
if dev_tbl.RMSE_vs_zero.iloc[0] >= 1:
    print("WARNING: even the best model does not beat the zero baseline on development data; the demo output is illustrative only.")

y = Y[DEMO_H].values
last_lab = n - DEMO_H                                  # rows 0..last_lab-1 have an observed label
val_hi, val_lo = last_lab, last_lab - VAL_DAYS
tr_hi = val_lo - DEMO_H
best_p, best_val = select_params(demo_name, Xv[:tr_hi], y[:tr_hi], Xv[val_lo:val_hi], y[val_lo:val_hi])
final_model = build(demo_name, best_p, y[:last_lab].std()).fit(Xv[:last_lab], y[:last_lab])
print("Chosen hyperparameters:", best_p, "| validation RMSE:", round(best_val, 6))

def _ver(p):
    try: return md_.version(p)
    except Exception: return "n/a"
MODEL_INFO = dict(
    model=demo_name, horizon_trading_days=DEMO_H, hyperparameters=best_p,
    selection_rule="lowest development-period RMSE vs zero baseline at the shortest horizon (holdout not used)",
    trained_rows=int(last_lab), trained_through_feature_date=str(dates[last_lab - 1].date()),
    target=TARGET, series_tickers={k: TICKER_OF[k] for k in kept}, n_lags=N_LAGS, lag_start=LAG_START, feature_columns=FEATURES,
    preprocessing="StandardScaler inside the sklearn pipeline for linear models (fitted on training data); none for tree models",
    missing_data_policy=dict(max_start_date=MAX_START_DATE, ffill_limit_days=5, excluded=excluded, prefilled_zero_returns=False),
    data_range=[str(prices.index.min().date()), str(prices.index.max().date())],
    versions={p: _ver(p) for p in ["scikit-learn", "xgboost", "lightgbm", "pandas", "numpy", "yfinance"]},
    created=datetime.datetime.now().isoformat(timespec="seconds"))
joblib.dump(final_model, "models/final_model.joblib")
json.dump(FEATURES, open("models/feature_columns.json", "w"), indent=2)
json.dump(MODEL_INFO, open("models/model_info.json", "w"), indent=2, default=str)

est = final_model[-1] if hasattr(final_model, "steps") else final_model
if hasattr(est, "coef_"):
    coefs = pd.Series(est.coef_, index=FEATURES)
    print(f"Non-zero coefficients: {(coefs != 0).sum()} of {len(coefs)}")
    display(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(10).to_frame("coefficient (standardised X)"))
elif hasattr(est, "feature_importances_"):
    display(pd.Series(est.feature_importances_, index=FEATURES).sort_values(ascending=False).head(10).to_frame("importance"))

In [ ]:
# ---------- one function builds live features; it reuses prices_to_features (same code path as training) ----------
def features_from_closes(closes_dict, series_cols):
    missing = [c_ for c_ in series_cols if c_ not in closes_dict or closes_dict[c_] is None or len(closes_dict[c_]) == 0]
    if missing:
        raise ValueError(f"No data for required series {missing}; refusing to guess or zero-fill them.")
    master_ = closes_dict[VW].dropna().index
    px = ffill_align(closes_dict, master_, cols=series_cols)
    if px.iloc[-(N_LAGS + 5):].isna().any().any():
        raise ValueError("Recent prices contain gaps longer than the 5-day forward-fill limit.")
    f = prices_to_features(px).dropna()
    if f.empty:
        raise ValueError("Not enough history to build features.")
    assert list(f.columns) == FEATURES
    return f.iloc[[-1]]

def predict_from_saved(feature_row):
    mdl = joblib.load("models/final_model.joblib"); cols = json.load(open("models/feature_columns.json"))
    assert list(feature_row.columns) == cols, "feature order differs from training"
    return float(mdl.predict(feature_row[cols].values)[0])

# ---------- REPRODUCIBILITY TEST: rebuild the latest features from truncated raw closes and reload the saved model ----------
assert json.load(open("models/feature_columns.json")) == FEATURES
recent = {k: closes[k].iloc[-150:] for k in kept}
f_chk = features_from_closes(recent, kept)
assert np.allclose(f_chk.values, X_full.iloc[[-1]].values), "live feature construction differs from training features"
pred_saved = predict_from_saved(f_chk)
pred_mem = float(final_model.predict(X_full.iloc[[-1]].values)[0])
assert np.isclose(pred_saved, pred_mem)
AUDIT["final_model_reproducible"] = True
print(f"Reproducibility test passed: features rebuilt from raw closes match training features exactly; saved model reproduces the prediction ({pred_saved:+.4%}).")

latest = X_full.iloc[[-1]]
print(f"\nLatest feature date in dataset: {latest.index[0].date()}")
print(f"Predicted VW {DEMO_H}-day return: {pred_saved:+.4%} -> signal: {'LONG' if pred_saved > 0 else 'SHORT' if pred_saved < 0 else 'FLAT'}")
print("Experimental output, NOT a trading recommendation (see the holdout results above).")

In [ ]:
# OPTIONAL live demo (needs internet): fresh prices -> SAME feature code -> saved model. Fails loudly instead of guessing missing data.
def live_prediction():
    cl = {}
    for c_ in kept:
        df = yf.download(TICKER_OF[c_], period="6mo", interval="1d", auto_adjust=True, progress=False)
        if df is None or df.empty:
            raise ValueError(f"download failed for {TICKER_OF[c_]}")
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        s = df["Close"].dropna(); s.index = pd.to_datetime(s.index).tz_localize(None)
        cl[c_] = s
    f = features_from_closes(cl, kept)
    return f.index[0].date(), predict_from_saved(f)
try:
    d_, p_ = live_prediction()
    print(f"LIVE: features as of {d_} -> predicted VW {DEMO_H}-day return {p_:+.4%}")
except Exception as e:
    print("Live prediction unavailable right now:", type(e).__name__, "-", e, "\n(The saved-data prediction above is still valid.)")

## 11. Generate files (only if every audit check passed)
Result tables, charts, CSVs, model metadata, `README.md` and `requirements.txt` are written **only now**, after all checks above passed in this run.

In [ ]:
print("AUDIT SUMMARY")
for k, v in AUDIT.items(): print(f"  {'PASS' if v else 'FAIL'}  {k}")
need = ["feature_target_alignment", "walk_forward_no_leakage", "identical_dates_and_baseline", "backtest_unit_tests", "final_model_reproducible"]
if not (all(AUDIT.get(k) for k in need)):
    for d in ["results", "models"]:
        for f in os.listdir(d): os.remove(os.path.join(d, f))
    raise AssertionError("Audit failed: generated files removed. Fix the failing check and re-run.")

metrics_df.to_csv("results/prediction_metrics.csv", index=False)
dev_df.to_csv("results/trading_development_grid.csv", index=False)
hold_df.to_csv("results/trading_holdout.csv", index=False)
policy_df.to_csv("results/data_policy_report.csv", index=False)
win_df.to_csv("results/walk_forward_windows.csv", index=False)
pd.DataFrame([dict(feature_date=latest.index[0].date(), horizon=DEMO_H, model=demo_name, predicted_return=pred_saved)]).to_csv("results/latest_prediction.csv", index=False)
for name, fig in FIGS.items():
    fig.savefig(f"results/{name}.png", dpi=150, bbox_inches="tight")

req = "\n".join(f"{p}=={_ver(p)}" for p in ["yfinance", "pandas", "numpy", "scikit-learn", "xgboost", "lightgbm", "matplotlib", "joblib"]) + "\njupyter\n"
open("requirements.txt", "w").write(req)

readme = '''# Quantitative Trading with Machine Learning (UE24CS352A Mini-Project)

Predicting Volkswagen (@@TARGET@@) returns from lagged returns of its suppliers and market proxies; comparing Linear Regression,
Elastic Net, Decision Tree, XGBoost and LightGBM against a zero-return baseline; and back-testing a threshold long/short strategy
against buy-and-hold and cash. Based on R. M. Glawion, *Quantitative Trading with Machine Learning* (Stanford CS229).

## Team
- <Name 1, SRN>
- <Name 2, SRN>

## Setup
```bash
git clone <your-repo-url>
cd <repo>
pip install -r requirements.txt
jupyter notebook Quantitative_Trading_Final.ipynb      # then: Restart & Run All
```
Or open the notebook in Google Colab (Runtime -> Run all). Internet is needed on the first run (Yahoo Finance via `yfinance`);
prices are cached in `data/raw/`. Runtime is roughly 5-15 minutes.

## Method (summary)
- **Data:** daily adjusted closes, @@RANGE@@ (@@NOBS@@ trading days), series kept: @@KEPT@@.
- **Missing data:** no pre-listing returns are filled with zero. Series whose first price is after @@MAXSTART@@ are excluded
  (excluded: @@EXCL@@); exchange-holiday gaps of at most 5 days are forward-filled; the sample starts when all kept series have prices.
- **Features:** last @@NLAGS@@ daily returns of each series; `lag0` is the return on the prediction date t. EUR/USD is used from lag 1 only (its daily close is after VW's close).
- **Targets:** VW return from the close of t to the close of t+h, h in @@HORIZONS@@.
- **Evaluation:** walk-forward (5-year train, 1-year validation, 1-month test, expanding), a gap of h days between train/validation/test.
  The final ~year (@@HOLDOUT@@ days) is an untouched holdout: trading thresholds and the demo model are chosen on the development period only.
- **Backtest:** long/short/hold with threshold c; costs on entry, every position change and exit; compared with buy-and-hold and cash on identical dates.
- **Audit checks** (all must pass before files are generated): feature/target alignment, no-leakage perturbation test, backtest unit tests,
  identical dates for all models, and reproducibility of the saved model's prediction.

## Outputs (generated by the notebook)
- `results/prediction_metrics.csv`, `results/trading_development_grid.csv`, `results/trading_holdout.csv`, `results/data_policy_report.csv`,
  `results/walk_forward_windows.csv`, `results/latest_prediction.csv`, `results/*.png`
- `models/final_model.joblib`, `models/feature_columns.json`, `models/model_info.json`

## Disclaimer
Educational project; results are exploratory and not financial advice.
'''
for k, v in {"@@TARGET@@": TARGET, "@@RANGE@@": f"{prices.index.min().date()} to {prices.index.max().date()}", "@@NOBS@@": str(len(prices)),
             "@@KEPT@@": ", ".join(TICKER_OF[k] for k in kept), "@@MAXSTART@@": MAX_START_DATE,
             "@@EXCL@@": (", ".join(TICKER_OF[k] for k in excluded) or "none"), "@@NLAGS@@": str(N_LAGS),
             "@@HORIZONS@@": str(HORIZONS), "@@HOLDOUT@@": str(HOLDOUT_DAYS)}.items():
    readme = readme.replace(k, v)
# Comprehensive README.md is maintained in repo root
pass
print("\nAll checks passed. Files written:")
for d in ["results", "models"]:
    print(f"  {d}/ ->", sorted(os.listdir(d)))
print("  README.md, requirements.txt")
# Automatically compile writeup PDF
try:
    import sys
    sys.path.append('.')
    from reports.generate_pdf import build_pdf
    build_pdf('reports/UE24CS352A_MiniProject_Writeup.pdf')
    print('PASS: Recompiled writeup PDF at reports/UE24CS352A_MiniProject_Writeup.pdf')
except Exception as e:
    print('PDF build notice:', e)
